# Eigen-CAM visualisations (Fig. 2)

Part of **FL-YOLO-DADP** (Chen et al., CDSR 2025). Produces the panels of Fig. 2: Eigen-CAM heat maps (on layer `model.model[-4]`) of YOLOv8n detectors trained on real frames, federated without/with DADP, and on each generator's synthetic frames, together with the synthetic version of the same frame.

Eigen-CAM comes from [rigvedrs/YOLO-V11-CAM](https://github.com/rigvedrs/YOLO-V11-CAM) (MIT licence), vendored unmodified in `third_party/yolo_cam`.

| panel | cell loads | produced by |
|---|---|---|
| 1, 2 | `../runs/baselines/local_agent2_epoch20` | `baselines/train_yolo.py` on agent_2, 20 epochs |
| 3 | `../runs/fl/FedMedian_3agents_round20_epoch1`, round 20 | `fl/run_fl.py --strategy FedMedian` |
| 4 | `../runs/fl/FedMedian_DADP_round20_epoch1_nm0_cn10`, round 12 | `fl/run_fl.py --dp dadp --noise-multiplier 0 --clipping-norm 10` |
| 5–12 | generator checkpoints + `../runs/baselines/synthetic_<method>` | `generative_models/*.ipynb`, then `baselines/train_yolo.py` |



In [ ]:
import sys
sys.path.insert(0, "third_party")  # vendored yolo_cam (rigvedrs/YOLO-V11-CAM, MIT)

# Stable Diffusion v1.5 for the ControlNet panel. The paper used "runwayml/stable-diffusion-v1-5"
# (since removed from the Hub); this is the official mirror of the same weights.
SD_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"


### https://github.com/rigvedrs/YOLO-V11-CAM

### original image

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

model = YOLO('../runs/baselines/local_agent2_epoch20/weights/best.pt')
model = model.cpu()

img = cv2.imread('../data/generator_train/test/images/test-agg1-2067.png')
rgb_img = img.copy()
print(np.min(rgb_img), np.max(rgb_img))
img = np.float32(img) / 255
print(np.min(img), np.max(img))

def add_scale_bar_pil(image, pixel_length=100, scale_text='100μm', position='lower right', 
                     font_path='arial.ttf', font_size=18):
    """
    Add scale bar with Unicode support using PIL
    :param image: Input RGB image (numpy array)
    :param pixel_length: Length of scale bar in pixels
    :param scale_text: Text to display with scale bar (supports Unicode)
    :param position: Position of scale bar
    :param font_path: Path to TrueType font file
    """
    # Convert to PIL Image
    pil_img = Image.fromarray(image)
    draw = ImageDraw.Draw(pil_img)
    
    # Load font
    try:
        font = ImageFont.truetype(font_path, font_size)
    except IOError:
        font = ImageFont.load_default()
    
    # Calculate positions
    w, h = pil_img.size
    if 'lower right' in position:
        start_x = w - pixel_length - 30
        start_y = h - 40
    else:  # lower left
        start_x = 20
        start_y = h - 40
    
    # Draw scale bar
    draw.rectangle([(start_x, start_y), 
                   (start_x + pixel_length, start_y + 1)],
                   fill='white', width=1)
    
    # Calculate text position
    text_bbox = draw.textbbox((0, 0), scale_text, font=font)
    text_width = text_bbox[2] - text_bbox[0]
    text_height = text_bbox[3] - text_bbox[1]
    
    # Draw text background
    draw.rectangle([(start_x, start_y - text_height - 10),
                   (start_x + text_width, start_y - 5)],
                   fill='black')
    
    # Draw text
    draw.text((start_x, start_y - text_height - 10), 
             scale_text, font=font, fill='white')
    
    return np.array(pil_img)

# Add scale bar with Unicode characters
# NOTE: Replace 'arial.ttf' with a font file that supports your characters
scaled_rgb_img = add_scale_bar_pil(
    rgb_img,
    pixel_length=50,
    scale_text='50μm',  # Now supports μ character
    position='lower right',
    font_path='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'  # Example Linux font path
)

target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(rgb_img)[0, :, :]
cam_image = show_cam_on_image(img, grayscale_cam, use_rgb=True)
# plt.imshow(cam_image)
scaled_rgb_img = cv2.cvtColor(scaled_rgb_img, cv2.COLOR_RGB2BGR)
print(np.min(scaled_rgb_img), np.max(scaled_rgb_img))
Image.fromarray(np.hstack((scaled_rgb_img, cam_image)))

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

model = YOLO('../runs/fl/FedMedian_3agents_round20_epoch1/round20/FL_aggregated_round_20.pt')
model = model.cpu()

img = cv2.imread('../data/generator_train/test/images/test-agg1-2067.png')
rgb_img = img.copy()
print(np.min(rgb_img), np.max(rgb_img))
img = np.float32(img) / 255
print(np.min(img), np.max(img))

target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(rgb_img)[0, :, :]
cam_image = show_cam_on_image(img, grayscale_cam, use_rgb=True)
plt.axis("off")
plt.imshow(cam_image)
# rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR)
# print(np.min(rgb_img), np.max(rgb_img))
# Image.fromarray(np.hstack((rgb_img, cam_image)))

### FL

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

experiment = 'FedMedian_DADP_round20_epoch1_nm0_cn10'
model = YOLO(f'../runs/fl/{experiment}/round12/FL_aggregated_round_12.pt')
model = model.cpu()

img = cv2.imread('../data/generator_train/test/images/test-agg1-2067.png')
rgb_img = img.copy()
print(np.min(rgb_img), np.max(rgb_img))
img = np.float32(img) / 255
print(np.min(img), np.max(img))

target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(rgb_img)[0, :, :]
cam_image = show_cam_on_image(img, grayscale_cam, use_rgb=True)
plt.axis("off")
plt.imshow(cam_image)

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

model = YOLO('../runs/fl/FedMedian_3agents_round20_epoch1/round20/FL_aggregated_round_20.pt')
model = model.cpu()

img = cv2.imread('../data/generator_train/test/images/test-agg1-2067.png')
rgb_img = img.copy()
print(np.min(rgb_img), np.max(rgb_img))
img = np.float32(img) / 255
print(np.min(img), np.max(img))

target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(rgb_img)[0, :, :]
cam_image = show_cam_on_image(img, grayscale_cam, use_rgb=True)
plt.axis("off")
plt.imshow(cam_image)
# rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR)
# print(np.min(rgb_img), np.max(rgb_img))
# Image.fromarray(np.hstack((rgb_img, cam_image)))

### encoder-decoder

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

device = "cuda"  # or "cpu"

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.conv(x)

class UNetEncoderDecoder(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super(UNetEncoderDecoder, self).__init__()
        
        # Downsampling path
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature
        
        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        
        # Upsampling path
        rev_features = features[::-1]
        for feature in rev_features:
            self.ups.append(
                nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)
            )
            self.ups.append(DoubleConv(feature*2, feature))
        
        # Final output layer
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x):
        # Encoder
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder
        skip_connections = skip_connections[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]
            
            # Concatenate skip connection
            x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](x)
        
        return self.final_conv(x)

# Load model (ensure UNetEncoderDecoder is defined or imported)
model = UNetEncoderDecoder(in_channels=3, out_channels=3).to(device)
model.load_state_dict(torch.load("../runs/generative_models/encoder-decoder/encoder_decoder_size512-384_feature512_best.pth", map_location=device))
model.eval()

# Read a test image (currently used as if it were the 3-channel mask)
test_image = cv2.imread('../data/generator_train/test/images/test-agg1-2067.png')
# Convert from BGR to RGB
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
# Convert to float and scale to [0,1]
test_image = test_image.astype(np.float32) / 255.0

# [H, W, C] --> [C, H, W], then add batch dimension
mask_tensor = torch.from_numpy(np.transpose(test_image, (2, 0, 1))).unsqueeze(0).to(device)

# Generate synthetic image
with torch.no_grad():
    fake_image = model(mask_tensor)  # shape: [1, 3, H, W]

# Convert to NumPy for display
fake_image_np = fake_image.squeeze(0).cpu().numpy()  # shape: [3, H, W]
fake_image_np = np.transpose(fake_image_np, (1, 2, 0))  # shape: [H, W, 3]
fake_image_np = np.clip(fake_image_np, 0, 1)
gray = fake_image_np.mean(axis=2, keepdims=True)  # shape (H,W,1)
fake_image_np = np.repeat(gray, 3, axis=2)   # shape (H,W,3)
print('out_np.shape', fake_image_np.shape)  # Should be [H, W, 3]
print('out_np min max values', np.min(fake_image_np), np.max(fake_image_np))

identical_across_channels = (fake_image_np[:,:,0] == fake_image_np[:,:,1]).all() \
                            and (fake_image_np[:,:,1] == fake_image_np[:,:,2]).all()
if identical_across_channels:
    print("Now all channels are identical across each pixel.")
else:
    print("Something went wrong; channels differ.")
out_endecoder=fake_image_np
plt.imshow(fake_image_np)
plt.axis("off")
plt.show()

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

experiment = 'encoder-decoder'
model = YOLO(f'../runs/baselines/synthetic_{experiment}/weights/best.pt')
model = model.cpu()

out_endecoder_rgb = np.float32(out_endecoder) * 255
print(np.min(out_endecoder_rgb), np.max(out_endecoder_rgb))
print(np.min(out_endecoder), np.max(out_endecoder))
target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(out_endecoder_rgb)[0, :, :]
cam_image = show_cam_on_image(out_endecoder, grayscale_cam, use_rgb=True)
# plt.imshow(cam_image)
# plt.show()
out_endecoder_rgb = cv2.cvtColor(out_endecoder_rgb.astype(np.uint8), cv2.COLOR_RGB2BGR)
print(np.min(out_endecoder_rgb), np.max(out_endecoder_rgb))
Image.fromarray(np.hstack((out_endecoder_rgb, cam_image)))

### GANs

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------
# 1. Define your Pix2PixGenerator
# ------------------------------
class DownBlock(nn.Module):
    """A single down block (Conv -> BN -> LeakyReLU) with stride=2."""
    def __init__(self, in_ch, out_ch, apply_norm=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        ]
        if apply_norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    """
    A single up block:
      1) TransposeConv (in -> out)
      2) (option) concat skip
      3) Conv (out + skip_ch -> out)
      4) (option) dropout
    This ensures channel counts match properly.
    """
    def __init__(self, in_ch, out_ch, skip_ch=0, dropout=False):
        super().__init__()
        self.up_conv = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
        self.up_bn = nn.BatchNorm2d(out_ch)
        
        # after we upsample and possibly concat skip, we do a 3x3 conv to get back to out_ch
        self.post_conv = nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1)
        self.post_bn = nn.BatchNorm2d(out_ch)
        
        self.relu = nn.ReLU(True)
        self.dropout = nn.Dropout(0.5) if dropout else nn.Identity()

    def forward(self, x, skip=None):
        """
        x: upsampled input, shape=[B, in_ch, H, W]
        skip: skip-connection features, shape=[B, skip_ch, H, W], or None
        """
        # 1) TransposeConv
        x = self.up_conv(x)
        x = self.up_bn(x)
        x = self.relu(x)
        
        # 2) Optionally concat skip
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        
        # 3) 3x3 conv to reduce channels back to out_ch
        x = self.post_conv(x)
        x = self.post_bn(x)
        x = self.relu(x)
        
        # 4) dropout if needed
        x = self.dropout(x)
        return x

class Pix2PixGenerator(nn.Module):
    """
    A 6-layer pix2pix generator that correctly aligns channel counts
    for (384,512) input. Each up block is explicitly defined with its
    skip input channel. No final skip to the input.
    """
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        
        # ----------------
        # 1) Down blocks
        # ----------------
        # d1 out = 64
        self.d1 = DownBlock(in_channels, 64,  apply_norm=False)
        # d2 out = 128
        self.d2 = DownBlock(64, 128)
        # d3 out = 256
        self.d3 = DownBlock(128, 256)
        # d4 out = 512
        self.d4 = DownBlock(256, 512)
        # d5 out = 512
        self.d5 = DownBlock(512, 512)
        # d6 out = 512 (no skip above this)
        self.d6 = DownBlock(512, 512, apply_norm=False)

        # ----------------
        # 2) Up blocks
        # ----------------
        # up1: in=512, skip=512 -> out=512
        self.u1 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        # up2: in=512, skip=512 -> out=512
        self.u2 = UpBlock(in_ch=512, out_ch=512, skip_ch=512, dropout=True)
        # up3: in=512, skip=256 -> out=256
        self.u3 = UpBlock(in_ch=512, out_ch=256, skip_ch=256, dropout=True)
        # up4: in=256, skip=128 -> out=128
        self.u4 = UpBlock(in_ch=256, out_ch=128, skip_ch=128)
        # up5: in=128, skip=64 -> out=64
        self.u5 = UpBlock(in_ch=128, out_ch=64, skip_ch=64)
        # up6: in=64, skip=None -> out=64
        self.u6 = UpBlock(in_ch=64, out_ch=64, skip_ch=0, dropout=False)

        # final 1x1 conv
        self.final = nn.Sequential(
            nn.Conv2d(64, out_channels, kernel_size=1),
            nn.Tanh()
        )

    def forward(self, x):
        # 1) Encoder
        d1_out = self.d1(x)    # 64
        d2_out = self.d2(d1_out)   # 128
        d3_out = self.d3(d2_out)   # 256
        d4_out = self.d4(d3_out)   # 512
        d5_out = self.d5(d4_out)   # 512
        d6_out = self.d6(d5_out)   # 512

        # 2) Decoder
        # up1 => skip with d5 => cat(512,512)=1024 => out=512
        u1_out = self.u1(d6_out, d5_out)
        # up2 => skip with d4 => cat(512,512)=1024 => out=512
        u2_out = self.u2(u1_out, d4_out)
        # up3 => skip with d3 => cat(512,256)=768 => out=256
        u3_out = self.u3(u2_out, d3_out)
        # up4 => skip with d2 => cat(256,128)=384 => out=128
        u4_out = self.u4(u3_out, d2_out)
        # up5 => skip with d1 => cat(128,64)=192 => out=64
        u5_out = self.u5(u4_out, d1_out)
        # up6 => no skip => out=64
        u6_out = self.u6(u5_out, None)

        return self.final(u6_out)

class PatchDiscriminator(nn.Module):
    """
    PatchGAN discriminator from pix2pix:
    Input: concatenation of mask and image -> shape = [N, (in_channels + out_channels), H, W].
    Output: real/fake patch logits, shape ~ [N, 1, H/16, W/16].
    """
    def __init__(self, in_channels=6): 
        # By default: mask(3 ch) + image(3 ch) = 6
        super().__init__()
        
        def disc_block(in_feat, out_feat, norm=True):
            layers = [nn.Conv2d(in_feat, out_feat, 4, stride=2, padding=1)]
            if norm:
                layers.append(nn.BatchNorm2d(out_feat))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        
        self.model = nn.Sequential(
            *disc_block(in_channels, 64, norm=False),  # (6 -> 64)
            *disc_block(64, 128),
            *disc_block(128, 256),
            # stride=1 for the last layer (scale down less)
            nn.Conv2d(256, 512, 4, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, stride=1, padding=1)  # Output channels = 1
        )

    def forward(self, x):
        return self.model(x)

# ------------------------------
# 2. Instantiate generator & load weights
# ------------------------------
model = Pix2PixGenerator(in_channels=3, out_channels=3).to(device)
checkpoint_path = "../runs/generative_models/GANs/models/20epochs/generator_best.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

# ------------------------------
# 3. Prepare test input
#    *IMPORTANT*: Must match the training resolution (256x256)
# ------------------------------
test_image_path = "../data/generator_train/test/images/test-agg1-2067.png"
test_image = cv2.imread(test_image_path)
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)

# Resize to 256x256 (or the resolution used in training)
test_image = cv2.resize(test_image, (512, 384), interpolation=cv2.INTER_AREA)

# Convert to float32, scale to [0,1]
test_image = test_image.astype(np.float32) / 255.0

# [H, W, C] -> [C, H, W] -> add batch dim -> to device
test_tensor = torch.from_numpy(np.transpose(test_image, (2, 0, 1))).unsqueeze(0).to(device)

# ------------------------------
# 4. Forward pass
# ------------------------------
with torch.no_grad():
    # The generator typically outputs in [-1, 1] range
    out = model(test_tensor)

# Convert from [-1,1] to [0,1] for visualization
out = (out + 1) / 2.0
out = torch.clamp(out, 0, 1)

out_np = out.squeeze(0).permute(1, 2, 0).cpu().numpy()  # [H,W,3]
gray = out_np.mean(axis=2, keepdims=True)  # shape (H,W,1)
out_np = np.repeat(gray, 3, axis=2)   # shape (H,W,3)
print('out_np.shape', out_np.shape)  # Should be [H, W, 3]
print('out_np min max values', np.min(out_np), np.max(out_np))

identical_across_channels = (out_np[:,:,0] == out_np[:,:,1]).all() \
                            and (out_np[:,:,1] == out_np[:,:,2]).all()
if identical_across_channels:
    print("Now all channels are identical across each pixel.")
else:
    print("Something went wrong; channels differ.")

out_gans = out_np
plt.imshow(out_gans)
# plt.title("Synthesized Image (Generator Output)")
plt.axis("off")
plt.show()

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

method = 'GANs'
model = YOLO(f'../runs/baselines/synthetic_{method}/weights/best.pt')
model = model.cpu()

out_gans_rgb = np.float32(out_gans) * 255
print(np.min(out_gans_rgb), np.max(out_gans_rgb))
print(np.min(out_gans), np.max(out_gans))
target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(out_gans_rgb)[0, :, :]
cam_image = show_cam_on_image(out_gans, grayscale_cam, use_rgb=True)
# plt.imshow(out_gans)
# plt.show()
out_gans_rgb = cv2.cvtColor(out_gans_rgb.astype(np.uint8), cv2.COLOR_RGB2BGR)
Image.fromarray(np.hstack((out_gans_rgb, cam_image)))

### DDPM

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

class SimpleConditionalUNet(nn.Module):
    def __init__(self, in_channels=6, base_channels=64):
        """
        in_channels = 6 => (3 for noisy image + 3 for mask)
        out_channels = 3 => predict noise for the (RGB) image
        """
        super().__init__()
        self.enc1 = self.conv_block(in_channels, base_channels)
        self.enc2 = self.conv_block(base_channels, base_channels*2, down=True)
        self.enc3 = self.conv_block(base_channels*2, base_channels*4, down=True)
        
        self.middle = self.conv_block(base_channels*4, base_channels*4, down=False)
        
        self.dec3 = self.conv_block(base_channels*8, base_channels*2, up=True)
        self.dec2 = self.conv_block(base_channels*4, base_channels, up=True)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_channels, 3, kernel_size=1),  # Output = 3 channels
        )

    def conv_block(self, in_c, out_c, down=False, up=False):
        """
        down=True -> stride=2 conv
        up=True   -> stride=2 transpose conv
        else standard conv
        """
        if down:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        elif up:
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.ReLU()
            )
        else:
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, 1, 1),
                nn.ReLU()
            )

    def forward(self, x, mask):
        """
        x : [B, 3, H, W] (noisy image)
        mask : [B, 3, H, W]
        """
        # Condition by concatenating
        c = torch.cat([x, mask], dim=1)  # [B, 6, H, W]
        e1 = self.enc1(c)               # [B, base, H, W]
        e2 = self.enc2(e1)              # [B, base*2, H/2, W/2]
        e3 = self.enc3(e2)              # [B, base*4, H/4, W/4]
        
        m = self.middle(e3)            # [B, base*4, H/4, W/4]
        
        d3 = self.dec3(torch.cat([m, e3], dim=1))  # skip conn
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        out = self.dec1(torch.cat([d2, e1], dim=1))
        return out

# Load the trained model
model = SimpleConditionalUNet(in_channels=6, base_channels=64).to(device)
checkpoint = "../runs/generative_models/DDPM/model/20epochs/diffusion_model_best.pth"
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()  # Set model to evaluation mode

# Beta schedule
def make_beta_schedule(start=1e-4, end=0.02, num_timesteps=1000):
    return torch.linspace(start, end, num_timesteps)  # Creates a linearly spaced schedule of betas.

num_timesteps = 200  # Match your code
betas = make_beta_schedule(start=1e-4, end=0.02, num_timesteps=num_timesteps).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def sample_diffusion(model, mask, betas, device):
    """
    Given a bounding-box mask, sample a random noise image and iteratively
    denoise it from T down to 0 using the model's predicted noise.
    
    mask: [1, 3, H, W]
    """
    model.eval()
    with torch.no_grad():
        num_timesteps = betas.shape[0]
        x = torch.randn_like(mask).to(device)  # x_T ~ N(0,I), shape = [1,3,H,W]

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        
        for i in reversed(range(num_timesteps)):
            t = torch.tensor([i], device=device, dtype=torch.long)
            # Predict noise
            pred_noise = model(x, mask)
            
            # x_{t-1} = 1/sqrt(alpha_t) ( x_t - (1 - alpha_t)/sqrt(1 - alpha_cumprod_t) * pred_noise ) + ...
            alpha_t = alphas[t]
            alpha_cumprod_t = alphas_cumprod[t]
            beta_t = betas[t]
            
            coef1 = 1.0 / torch.sqrt(alpha_t)
            coef2 = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_cumprod_t)
            
            x = coef1 * ( x - coef2 * pred_noise )
            
            # Add noise, except at i=0
            if i > 0:
                z = torch.randn_like(x)
                sigma = torch.sqrt(beta_t)
                x = x + sigma * z
        
        return x  # x_0

# Load a test mask image
test_mask_path = "../data/generator_train/test/images/test-agg1-2067.png"
mask_img = cv2.imread(test_mask_path)
mask_img = cv2.cvtColor(mask_img, cv2.COLOR_BGR2RGB)
mask_img = cv2.resize(mask_img, (512, 384), interpolation=cv2.INTER_NEAREST)
mask_img = mask_img.astype(np.float32) / 255.0  # Normalize to [0, 1]
mask_tensor = torch.from_numpy(np.transpose(mask_img, (2, 0, 1))).unsqueeze(0).to(device)  # [1,3,H,W]

# Generate the image
x_0 = sample_diffusion(model, mask_tensor, betas, device)  # shape [1,3,H,W]

# Clamp the output to [0, 1] for visualization
x_0 = torch.clamp(x_0, 0.0, 1.0)

# Convert to numpy for visualization
out_np = x_0.squeeze(0).permute(1, 2, 0).cpu().numpy()  # [H,W,3]
gray = out_np.mean(axis=2, keepdims=True)  # shape (H,W,1)
out_np = np.repeat(gray, 3, axis=2)   # shape (H,W,3)

# 4) Check channel identity
identical_across_channels = (out_np[:,:,0] == out_np[:,:,1]).all() \
                            and (out_np[:,:,1] == out_np[:,:,2]).all()
if identical_across_channels:
    print("Now all channels are identical across each pixel.")
else:
    print("Something went wrong; channels differ.")

print('out_np.shape', out_np.shape)  # Should be [H, W, 3]
print('out_np min max values', np.min(out_np), np.max(out_np))

out_ddpm = out_np
plt.imshow(out_ddpm)
plt.axis("off")
plt.show()

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

method = 'DDPM'
model = YOLO(f'../runs/baselines/synthetic_{method}/weights/best.pt')
model = model.cpu()

out_ddpm_rgb = np.float32(out_ddpm) * 255
print(np.min(out_ddpm_rgb), np.max(out_ddpm_rgb))
print(np.min(out_ddpm), np.max(out_ddpm))
target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(out_ddpm_rgb)[0, :, :]
cam_image = show_cam_on_image(out_ddpm, grayscale_cam, use_rgb=True)
# plt.imshow(cam_image)
# plt.show()
out_ddpm_rgb = cv2.cvtColor(out_ddpm_rgb.astype(np.uint8), cv2.COLOR_RGB2BGR)
Image.fromarray(np.hstack((out_ddpm_rgb, cam_image)))

### controlnet-latent-img2img

In [ ]:
import os
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from diffusers import UNet2DConditionModel, AutoencoderKL, ControlNetModel, DDPMScheduler

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_trained_controlnet(model_path, device):
    controlnet = ControlNetModel.from_pretrained(model_path).to(device)
    controlnet.eval()
    return controlnet

def load_pretrained_models(device):
    vae = AutoencoderKL.from_pretrained(
        SD_MODEL_ID,
        subfolder="vae",
        torch_dtype=torch.float32
    ).to(device)
    vae.eval()
    
    unet = UNet2DConditionModel.from_pretrained(
        SD_MODEL_ID,
        subfolder="unet",
        torch_dtype=torch.float32
    ).to(device)
    unet.eval()
    
    return vae, unet

def preprocess_image(image_path, size=(512, 384)):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(size, Image.BILINEAR)
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        T.Lambda(lambda x: torch.clamp(x, -1, 1))
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)
    return image_tensor

def synthesize_image(controlnet, vae, unet, noise_scheduler, image_tensor, num_inference_steps=50):
    controlnet.eval()
    vae.eval()
    unet.eval()
    
    with torch.no_grad():
        # 1) Encode the input image into latents
        latents = vae.encode(image_tensor).latent_dist.sample() * 0.18215
        
        # 2) We use the same image as the "control" (though typically you'd pass a bounding-box mask).
        control_cond = image_tensor
        
        # 3) Dummy text embeddings (all zeros)
        context_embeds = torch.zeros((1, 77, 768), device=device, dtype=torch.float32)
        
        # 4) Sampling steps (partial, only 'num_inference_steps' from the start)
        for t in noise_scheduler.timesteps[:num_inference_steps]:
            cnet_out = controlnet(
                sample=latents,
                timestep=t,
                encoder_hidden_states=context_embeds,
                controlnet_cond=control_cond,
                return_dict=True
            )
            control_pred = cnet_out.down_block_res_samples + [cnet_out.mid_block_res_sample]
            
            unet_out = unet(
                sample=latents,
                timestep=t,
                encoder_hidden_states=context_embeds,
                down_block_additional_residuals=control_pred[:-1],
                mid_block_additional_residual=control_pred[-1],
                return_dict=True
            )
            # Update latents
            latents = noise_scheduler.step(unet_out.sample, t, latents).prev_sample
        
        # 5) Decode latents -> image
        synthesized_image = vae.decode(latents / 0.18215).sample
        synthesized_image = torch.clamp(synthesized_image, -1, 1)
        synthesized_image = (synthesized_image + 1) / 2.0  # [0,1]
        # shape: [1, 3, H, W]

        # Convert to NumPy for post-processing
        synthesized_image = synthesized_image.squeeze(0).permute(1, 2, 0).cpu().numpy()  # (H, W, 3)
        
        return synthesized_image

model_path = "../runs/generative_models/controlnet_img2img/model/20epochs/controlnet_best.pth"
controlnet = load_trained_controlnet(model_path, device)
vae, unet = load_pretrained_models(device)

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_start=0.00085,
    beta_end=0.012,
    beta_schedule="scaled_linear",
    clip_sample=False
)

# 1) Load your test grayscale image
test_image_path = "../data/generator_train/test/images/test-agg1-2067.png"
image_tensor = preprocess_image(test_image_path)

# 2) Generate
synthesized_image = synthesize_image(controlnet, vae, unet, noise_scheduler, image_tensor)
print(f"synthesized_image shape: {synthesized_image.shape}, range=({synthesized_image.min():.4f},{synthesized_image.max():.4f})")

# 3) Force all channels identical
#    We'll average across the 3 channels and replicate back:
gray = synthesized_image.mean(axis=2, keepdims=True)  # shape (H,W,1)
synthesized_image_gray = np.repeat(gray, 3, axis=2)   # shape (H,W,3)

# 4) Check channel identity
identical_across_channels = (synthesized_image_gray[:,:,0] == synthesized_image_gray[:,:,1]).all() \
                            and (synthesized_image_gray[:,:,1] == synthesized_image_gray[:,:,2]).all()
if identical_across_channels:
    print("Now all channels are identical across each pixel.")
else:
    print("Something went wrong; channels differ.")

# 5) Plot
plt.imshow(synthesized_image_gray, cmap='gray')  # channels are same
plt.axis("off")
plt.show()

In [ ]:
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

method = 'controlnet_img2img'
model = YOLO(f'../runs/baselines/synthetic_{method}/weights/best.pt')
model = model.cpu()

synthesized_image_gray_rgb = np.float32(synthesized_image_gray) * 255
print(np.min(synthesized_image_gray_rgb), np.max(synthesized_image_gray_rgb))
print(np.min(synthesized_image_gray), np.max(synthesized_image_gray))
target_layers =[model.model.model[-4]]

cam = EigenCAM(model, target_layers,task='od')
grayscale_cam = cam(synthesized_image_gray_rgb)[0, :, :]
cam_image = show_cam_on_image(synthesized_image_gray, grayscale_cam, use_rgb=True)
# plt.imshow(cam_image)
# plt.show()
synthesized_image_gray_rgb = cv2.cvtColor(synthesized_image_gray_rgb.astype(np.uint8), cv2.COLOR_RGB2BGR)
Image.fromarray(np.hstack((synthesized_image_gray_rgb, cam_image)))